# Synthetic datasets

## Exercise 4 (Regresssion)

**TODO**: maybe even simulate names/employee-ids?

In [1]:
import numpy as np
import pandas as pd
from faker import Faker

np.random.seed(42)
fake = Faker("de_DE")  # German-style names
Faker.seed(42)

n = 1400

departments = np.random.choice(["Sales", "IT", "HR"], size=n, p=[0.4, 0.4, 0.2])

# --- synthetic IDs ---
employee_id = [f"E{100000+i}" for i in range(n)]

# --- names via Faker ---
names = [fake.name() for _ in range(n)]

# --- roles depending on department ---
roles = []
for d in departments:
    if d == "IT":
        roles.append(np.random.choice([
            "Software Engineer", "DevOps Engineer", "Data Scientist",
            "IT Project Lead", "System Administrator", "Line Manager"
        ], p=[0.25, 0.15, 0.15, 0.15, 0.15, 0.15]))
    elif d == "Sales":
        roles.append(np.random.choice([
            "Sales Representative", "Account Manager", "Sales Lead",
            "Business Development Manager", "Key Account Manager", "Line Manager"
        ], p=[0.30, 0.20, 0.15, 0.15, 0.10, 0.10]))
    else:  # HR
        roles.append(np.random.choice([
            "HR Specialist", "Recruiter", "HR Business Partner",
            "People Operations Lead", "HR Manager", "Line Manager"
        ], p=[0.25, 0.20, 0.20, 0.15, 0.10, 0.10]))

# Core features
experience = np.random.randint(0, 21, n)
training_hours = np.random.normal(40, 10, n).clip(5, 80)
team_size = np.random.randint(3, 15, n)
remote_share = np.random.beta(2.2, 2.0, n)

# Manager quality
manager_quality = np.zeros(n, dtype=int)
for i, d in enumerate(departments):
    if d == "Sales":
        manager_quality[i] = np.random.choice([1, 2, 3, 4, 5], p=[0.12, 0.23, 0.30, 0.22, 0.13])
    elif d == "IT":
        manager_quality[i] = np.random.choice([1, 2, 3, 4, 5], p=[0.04, 0.08, 0.18, 0.36, 0.34])
    else:
        manager_quality[i] = np.random.choice([1, 2, 3, 4, 5], p=[0.18, 0.28, 0.27, 0.17, 0.10])

salary = 30000 + experience * 2000 + np.random.normal(0, 5000, n)

dept_effect = {"Sales": 10, "IT": 5, "HR": 0}
dept_numeric = np.array([dept_effect[d] for d in departments])

training_c = training_hours - training_hours.mean()
remote_c = remote_share - remote_share.mean()

manager_c_within = np.zeros(n)
for d in ["Sales", "IT", "HR"]:
    idx = departments == d
    manager_c_within[idx] = manager_quality[idx] - manager_quality[idx].mean()

training_manager_interaction = 0.16 * training_c * manager_c_within

remote_main = -0.8 * remote_c

remote_dept_main = {"Sales": -4.0, "IT": 2.5, "HR": -5.0}
remote_dept_component = np.array(
    [remote_dept_main[d] for d in departments]
) * remote_c

remote_interaction_strength = {"Sales": 9.0, "IT": 12.0, "HR": 10.0}
remote_manager_interaction = np.array(
    [remote_interaction_strength[d] for d in departments]
) * remote_c * manager_c_within

manager_main_strength = {"Sales": 4.8, "IT": 6.8, "HR": 4.0}
manager_component = np.array(
    [manager_main_strength[d] for d in departments]
) * manager_quality

performance = (
    50
    + 1.8 * experience
    + 0.9 * training_hours
    + 0.0004 * salary
    + 1.4 * team_size
    + manager_component
    + remote_main
    + remote_dept_component
    + training_manager_interaction
    + remote_manager_interaction
    + dept_numeric
    + np.random.normal(0, 9, n)
)

df = pd.DataFrame({
    "employee_id": employee_id,
    "name": names,
    "role": roles,
    "department": departments,
    "performance": performance,
    "experience": experience,
    "training_hours": training_hours,
    "salary": salary,
    "remote_share": remote_share,
    "team_size": team_size,
    "manager_quality": manager_quality
})

df.to_csv("employee_performance_data.csv", index=False)

print(df.head())
print(df.groupby("department")[["manager_quality", "remote_share", "performance"]].mean())

  employee_id                name                          role department  \
0     E100000  Aleksandr Weihmann          Sales Representative      Sales   
1     E100001       Eleni Hauffer                 HR Specialist         HR   
2     E100002     Paul Dobes-Stey             Software Engineer         IT   
3     E100003      Klemens Löchel          System Administrator         IT   
4     E100004    Alexandre Davids  Business Development Manager      Sales   

   performance  experience  training_hours        salary  remote_share  \
0   198.242841          18       45.381330  59096.739149      0.227314   
1   144.281750           1       48.305080  34060.355685      0.419772   
2   191.788132          16       29.344653  64625.846566      0.782596   
3   190.445249          10       51.931709  38057.342688      0.763128   
4   173.408707          10       32.776373  45828.309067      0.499737   

   team_size  manager_quality  
0         11                5  
1          5          